# Phase 1 Data Audit: India-wide Crop Yield & Climate Dataset

**Project:** Rainfall-Lag-Aware Crop Yield Forecasting in India Using Explainable Temporal Machine Learning

**Phase 1 Objective:** Baseline empirical audit and feasibility analysis of historical district-level rice yield and aggregate meteorological records across India.

> **Important Data Limitation:** This raw dataset provides aggregate seasonal/annual rainfall and temperature records (`IMD_RF`, `IMD_Tmax`, `SD_RF_jun_nov`). It does **not** contain daily or weekly rainfall time series. Therefore, Phase 1 strictly audits the actual available variables and establishes baseline performance without manufacturing artificial rainfall lags.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure src is discoverable
sys.path.insert(0, str(Path.cwd()))

from src.data_loader import load_raw_data
from src.validation import validate_raw_schema, validate_key_integrity
from src.config import load_config

config = load_config()
print('Configuration loaded successfully.')

Configuration loaded successfully.


## 1. Load Raw Dataset
We load the raw CSV (`Data.csv` / `Data(1).csv`) using `latin1` encoding to handle non-breaking space bytes.

In [2]:
raw_df = load_raw_data()
print(f'Raw Dataset Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
validate_raw_schema(raw_df)
raw_df.head(5)

Raw Dataset Shape: 10279 rows, 15 columns


,Dist.Code,Sh_Dist_code,FID_1,State,AgroClimate,District,Year,RICE.AREA..1000.ha.,RICE.PRODUCTION..1000.tons.,RICE.YIELD..Kg.per.ha.,RICE.SHARE.IN.TOTAL..Percent.,Crops,IMD_RF,IMD_Tmax,SD_RF_jun_nov
0,1,13043,84,Chhattisgarh,Eastern Plateau and Hill Region,Durg,1990,397.9,481.4,1210.0,52.49,Rice,1440.475230,30.750970,13.091022
1,1,13043,84,Chhattisgarh,Eastern Plateau and Hill Region,Durg,1991,393.2,508.6,1293.0,51.85,Rice,1037.771550,31.712206,12.799949
2,1,13043,84,Chhattisgarh,Eastern Plateau and Hill Region,Durg,1992,398.4,514.5,1291.0,52.39,Rice,1135.394333,32.658029,15.601925
3,1,13043,84,Chhattisgarh,Eastern Plateau and Hill Region,Durg,1993,410.2,569.1,1387.0,51.51,Rice,1133.654012,32.109573,13.998169
4,1,13043,84,Chhattisgarh,Eastern Plateau and Hill Region,Durg,1994,430.1,601.7,1399.0,52.56,Rice,1646.387207,30.718793,18.795589


## 2. Columns and Data Types Audit
Inspect dtypes, memory usage, and column names.

In [3]:
dtypes_df = pd.DataFrame({
    'Column': raw_df.columns,
    'Dtype': raw_df.dtypes.values,
    'Non-Null Count': raw_df.notnull().sum().values,
    'Null Count': raw_df.isnull().sum().values,
    'Null Pct (%)': np.round(raw_df.isnull().mean().values * 100, 2)
})
dtypes_df

,Column,Dtype,Non-Null Count,Null Count,Null Pct (%)
0,Dist.Code,int64,10279,0,0.00
1,Sh_Dist_code,int64,10279,0,0.00
2,FID_1,int64,10279,0,0.00
3,State,str,10279,0,0.00
4,AgroClimate,str,10111,168,1.63
5,District,str,10279,0,0.00
6,Year,int64,10279,0,0.00
7,RICE.AREA..1000.ha.,float64,10279,0,0.00
8,RICE.PRODUCTION..1000.tons.,float64,10279,0,0.00
9,RICE.YIELD..Kg.per.ha.,float64,10279,0,0.00


## 3. Temporal Coverage & Year Summary
Investigate the distribution of records across calendar crop years (1990–2017).

In [4]:
year_counts = raw_df['Year'].value_counts().sort_index().reset_index()
year_counts.columns = ['Year', 'Observation_Count']
year_summary_path = Path('results/tables/year_summary.csv')
year_summary_path.parent.mkdir(parents=True, exist_ok=True)
year_counts.to_csv(year_summary_path, index=False)
print(f'Temporal Range: {raw_df["Year"].min()} to {raw_df["Year"].max()} ({raw_df["Year"].nunique()} unique years)')
print(f'Mean observations per year: {year_counts["Observation_Count"].mean():.1f}')
year_counts.head(10)

Temporal Range: 1990 to 2017 (28 unique years)
Mean observations per year: 367.1


,Year,Observation_Count
0,1990,375
1,1991,372
2,1992,372
3,1993,374
4,1994,370
5,1995,360
6,1996,363
7,1997,358
8,1998,359
9,1999,376


## 4. Geographic Coverage & Summary
Audit unique states, agro-climatic zones, and district names across India.

In [5]:
geo_summary = raw_df.groupby('State').agg(
    n_districts=('District', 'nunique'),
    n_dist_codes=('Dist.Code', 'nunique'),
    total_observations=('Year', 'count'),
    min_year=('Year', 'min'),
    max_year=('Year', 'max'),
    mean_rice_yield=('RICE.YIELD..Kg.per.ha.', 'mean')
).reset_index().sort_values(by='total_observations', ascending=False)

geo_summary['mean_rice_yield'] = np.round(geo_summary['mean_rice_yield'], 2)
geo_path = Path('results/tables/geography_summary.csv')
geo_summary.to_csv(geo_path, index=False)
print(f'Total States: {raw_df["State"].nunique()}')
print(f'Total Unique District Names: {raw_df["District"].nunique()}')
print(f'Total Unique Dist.Code: {raw_df["Dist.Code"].nunique()}')
geo_summary

Total States: 20


Total Unique District Names: 381
Total Unique Dist.Code: 384


,State,n_districts,n_dist_codes,total_observations,min_year,max_year,mean_rice_yield
17,UTTAR PRADESH,54,54,1512,1990,2017,2011.14
10,MADHYA PRADESH,38,38,1001,1990,2017,1034.90
2,BIHAR,30,30,810,1990,2017,1520.46
11,MAHARASHTRA,29,29,753,1990,2017,1238.19
1,ASSAM,23,23,644,1990,2017,1628.08
8,KARNATAKA,20,20,560,1990,2017,2395.89
15,TAMIL NADU,20,20,560,1990,2017,3481.11
14,RAJASTHAN,23,23,481,1990,2017,1458.41
19,WEST BENGAL,15,15,420,1990,2017,2380.53
5,HARYANA,15,15,404,1990,2017,2823.29


## 5. Missingness Analysis
Quantify missing values across all raw fields.

In [6]:
missing_df = pd.DataFrame({
    'Column': raw_df.columns,
    'Missing_Count': raw_df.isnull().sum().values,
    'Missing_Percentage': np.round(raw_df.isnull().mean().values * 100, 3)
}).sort_values(by='Missing_Count', ascending=False).reset_index(drop=True)

missing_path = Path('results/tables/missingness.csv')
missing_df.to_csv(missing_path, index=False)
missing_df

,Column,Missing_Count,Missing_Percentage
0,RICE.SHARE.IN.TOTAL..Percent.,758,7.374
1,AgroClimate,168,1.634
2,Sh_Dist_code,0,0.000
3,FID_1,0,0.000
4,State,0,0.000
5,District,0,0.000
6,Dist.Code,0,0.000
7,Year,0,0.000
8,RICE.AREA..1000.ha.,0,0.000
9,RICE.PRODUCTION..1000.tons.,0,0.000


## 6. Duplicate Analysis & Geographic Key Integrity
**Critical Investigation:** Do duplicate `District + Year` combinations exist? Why?

In [7]:
exact_dupes = raw_df.duplicated().sum()
dist_yr_dupes = raw_df.duplicated(subset=['District', 'Year']).sum()
state_dist_yr_dupes = raw_df.duplicated(subset=['State', 'District', 'Year']).sum()
code_yr_dupes = raw_df.duplicated(subset=['Dist.Code', 'Year']).sum()
state_code_yr_dupes = raw_df.duplicated(subset=['State', 'Dist.Code', 'Year']).sum()

key_report = pd.DataFrame([
    {'Key_Combination': 'Exact Full Row', 'Duplicates': exact_dupes, 'Is_Globally_Unique': exact_dupes == 0},
    {'Key_Combination': 'District + Year', 'Duplicates': dist_yr_dupes, 'Is_Globally_Unique': dist_yr_dupes == 0},
    {'Key_Combination': 'State + District + Year', 'Duplicates': state_dist_yr_dupes, 'Is_Globally_Unique': state_dist_yr_dupes == 0},
    {'Key_Combination': 'Dist.Code + Year', 'Duplicates': code_yr_dupes, 'Is_Globally_Unique': code_yr_dupes == 0},
    {'Key_Combination': 'State + Dist.Code + Year', 'Duplicates': state_code_yr_dupes, 'Is_Globally_Unique': state_code_yr_dupes == 0},
])

key_path = Path('results/tables/key_integrity_report.csv')
key_report.to_csv(key_path, index=False)
print(f'Exact Row Duplicates: {exact_dupes}')
print(f'District + Year Collisions: {dist_yr_dupes}')
print(f'State + Dist.Code + Year Collisions: {state_code_yr_dupes}')
key_report

Exact Row Duplicates: 0
District + Year Collisions: 76
State + Dist.Code + Year Collisions: 0


,Key_Combination,Duplicates,Is_Globally_Unique
0,Exact Full Row,0,True
1,District + Year,76,False
2,State + District + Year,0,True
3,Dist.Code + Year,0,True
4,State + Dist.Code + Year,0,True


### Inspecting Colliding Districts
Districts sharing identical names across multiple states:

In [8]:
dupe_dists = raw_df[raw_df.duplicated(subset=['District', 'Year'], keep=False)]
colliding_names = sorted(dupe_dists['District'].unique())
print(f'Districts appearing in multiple states ({len(colliding_names)}): {colliding_names}')
dupe_dists[['State', 'District', 'Dist.Code', 'Year', 'RICE.YIELD..Kg.per.ha.']].head(12)

Districts appearing in multiple states (3): ['Aurangabad', 'Bilaspur', 'Hamirpur']


,State,District,Dist.Code,Year,RICE.YIELD..Kg.per.ha.
84,Chhattisgarh,Bilaspur,4,1990,1398.000000
85,Chhattisgarh,Bilaspur,4,1991,1297.000000
86,Chhattisgarh,Bilaspur,4,1992,1047.000000
87,Chhattisgarh,Bilaspur,4,1993,1352.000000
88,Chhattisgarh,Bilaspur,4,1994,1618.000000
89,Chhattisgarh,Bilaspur,4,1995,1332.000000
90,Chhattisgarh,Bilaspur,4,1996,1341.000000
91,Chhattisgarh,Bilaspur,4,1997,496.000000
92,Chhattisgarh,Bilaspur,4,1998,851.333333
93,Chhattisgarh,Bilaspur,4,1999,1439.666667


## 7. Target and Climate Distribution Statistics
Examine min, median, mean, max, and check for negative values or zeros.

In [9]:
num_cols = ['RICE.YIELD..Kg.per.ha.', 'IMD_RF', 'IMD_Tmax', 'SD_RF_jun_nov']
stats_df = raw_df[num_cols].describe().T
stats_df['non_positive_count'] = [(raw_df[col] <= 0).sum() for col in num_cols]
stats_df

,count,mean,std,min,25%,50%,75%,max,non_positive_count
RICE.YIELD..Kg.per.ha.,10279.0,1936.297886,956.350302,44.000000,1205.000000,1874.000000,2527.000000,6316.000000,0
IMD_RF,10279.0,1026.339398,599.998710,28.291395,649.334933,903.223897,1228.498590,5229.077799,0
IMD_Tmax,10279.0,31.672977,2.017887,24.347551,30.379364,32.095441,33.039865,37.278471,0
SD_RF_jun_nov,10279.0,12.661560,5.173457,0.919056,9.217910,11.850967,15.042485,51.263938,0


## 8. Consolidated Dataset Summary Table
Save `results/tables/dataset_summary.csv` summarizing all fundamental attributes.

In [10]:
dataset_summary = pd.DataFrame([
    {'Metric': 'Total Raw Observations', 'Value': len(raw_df)},
    {'Metric': 'Total Columns', 'Value': raw_df.shape[1]},
    {'Metric': 'Start Year', 'Value': int(raw_df['Year'].min())},
    {'Metric': 'End Year', 'Value': int(raw_df['Year'].max())},
    {'Metric': 'Number of Unique Years', 'Value': int(raw_df['Year'].nunique())},
    {'Metric': 'Number of Unique States', 'Value': int(raw_df['State'].nunique())},
    {'Metric': 'Number of Unique District Names', 'Value': int(raw_df['District'].nunique())},
    {'Metric': 'Number of Unique District Codes', 'Value': int(raw_df['Dist.Code'].nunique())},
    {'Metric': 'Exact Row Duplicates', 'Value': int(exact_dupes)},
    {'Metric': 'Missing Target Observations', 'Value': int(raw_df['RICE.YIELD..Kg.per.ha.'].isnull().sum())},
    {'Metric': 'Missing Rainfall Observations', 'Value': int(raw_df['IMD_RF'].isnull().sum())},
    {'Metric': 'Non-Positive Target Count', 'Value': int((raw_df['RICE.YIELD..Kg.per.ha.'] <= 0).sum())},
])
ds_path = Path('results/tables/dataset_summary.csv')
dataset_summary.to_csv(ds_path, index=False)
dataset_summary

,Metric,Value
0,Total Raw Observations,10279
1,Total Columns,15
2,Start Year,1990
3,End Year,2017
4,Number of Unique Years,28
5,Number of Unique States,20
6,Number of Unique District Names,381
7,Number of Unique District Codes,384
8,Exact Row Duplicates,0
9,Missing Target Observations,0
